# Dil modeli ortamı: Colab'da şemaya zorlanmış çıkarma

Bu defter, standart PDF'lerinden test şartlarını **yerel çalışan** 7–8B, 4 bitlik bir dil modeliyle okumak ve
"yalnız dil modeli" yaklaşımını ölçmek içindir.

> **Önemli: Colab yalnız geliştirme ve ölçüm içindir.** Nihai sistem internetsiz, kendi bilgisayarında çalışır.
> Bu defterde internet yalnızca paketleri ve model dosyasını indirmek için kullanılır. Modelle konuşan kod
> (`src/malzeme_kabul/yerel_model.py`) yalnız `localhost`'a bağlanır; Colab'da da model aynı makinede çalışan
> Ollama ya da llama.cpp sunucusundadır, hiçbir dış API çağrılmaz.
>
> Colab'a yalnızca kamuya açık standartları (data/raw altındaki MIL-PRF PDF'leri) ve üretilmiş sentetik
> sertifikaları yükle. Gerçek tedarikçi sertifikalarını ve gizli belgeleri Colab'a yükleme.

**Dil modeli yalnız okur.** Model her sayfadan test adını, metodu, sınır değerlerini, birimi ve madde numarasını
`TestSarti` şemasına zorlanmış JSON olarak döner. Birim dönüşümü ve sınır karşılaştırması kesin kurallarla
Python'da yapılır (CLAUDE.md, kural 1).

Akış:
1. GPU'yu denetle, depoyu kur.
2. Ollama'yı kur, modeli indir (varsayılan yol). İstersen 7. bölümde llama.cpp ile aynı işi yap.
3. Tek sayfayla dene, sonra bütün belgeyi çıkar; belge başına süreyi kaydet.
4. Aday modelleri sırayla çalıştır, özet tabloyu karşılaştır.
5. Çıktıları indir.

Çalışma zamanı: **Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU**. T4'te 15 GB bellek var; 7–8B
modelin 4 bitlik (Q4_K_M) hali yaklaşık 5 GB tutar ve 8K bağlamla rahat sığar.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

## 1. Ayarlar

Model henüz seçilmedi. Aşağıdaki adaylar `docs/dil_modeli_ortami.md` içinde gerekçeleriyle anlatılıyor.
`MODEL` yalnızca ilk denemede kullanılacak adaydır; seçim ölçümden sonra yapılacak.

In [ ]:
REPO_URL = "https://github.com/nnidaozer/malzeme-kabul-ai.git"
DAL = "main"

# 7–8B, 4 bit (Ollama etiketleri; hepsi varsayılan olarak Q4_K_M nicemlemesi)
ADAYLAR_7_8B = {
    "qwen2.5:7b-instruct-q4_K_M": {"dusunme_kapali": False},
    "llama3.1:8b-instruct-q4_K_M": {"dusunme_kapali": False},
    "qwen3:8b": {"dusunme_kapali": True},   # Qwen3'ün düşünme kipi kapatılır
}
# Dizüstü adayları (1–3B). Doğruluk donanımdan bağımsız olduğu için F1'i burada ölçmek mümkün;
# süreyi ise dizüstünde ayrıca ölçmek gerekir.
ADAYLAR_1_3B = {
    "qwen2.5:3b-instruct-q4_K_M": {"dusunme_kapali": False},
    "llama3.2:3b-instruct-q4_K_M": {"dusunme_kapali": False},
    "qwen2.5:1.5b-instruct-q4_K_M": {"dusunme_kapali": False},
}

MODEL = "qwen2.5:7b-instruct-q4_K_M"   # ilk deneme adayı, seçim değil
BELGELER = ["data/raw/MIL-PRF-23377K.pdf"]   # gerekirse: "data/raw/MIL-PRF-81733D.pdf", "data/raw/MIL-PRF-85285E.pdf"
BAGLAM = 8192

## 2. Depoyu kur

Depo gizli olduğu için klonlama bir GitHub erişim belirteci (token) ister. Belirteç Colab'ın **Gizli anahtarlar**
(Secrets, sol menüdeki 🔑) bölümünde `GITHUB_TOKEN` adıyla durur; hücre onu oradan okur. Belirteç ekrana
yazdırılmaz, deftere ya da depoya (`.git/config`) kaydedilmez. Nasıl oluşturulacağı:
`docs/dil_modeli_ortami.md` → "Colab için GitHub belirteci".

Aşağıdaki üç hücreyi sırayla çalıştır. Çalışma zamanı yeniden başlatılırsa üçünü de tekrar çalıştır.

In [ ]:
# 2a. Depoyu Secrets'taki belirteçle klonla
import base64, os, subprocess, sys

DEPO_KLASORU = "/content/malzeme-kabul-ai"

if not os.path.isdir(os.path.join(DEPO_KLASORU, ".git")):
    from google.colab import userdata
    try:
        belirtec = userdata.get("GITHUB_TOKEN")
    except userdata.SecretNotFoundError:
        raise RuntimeError("Secrets'ta GITHUB_TOKEN yok. Sol menüde 🔑 → 'Add new secret' ile ekle.") from None
    except userdata.NotebookAccessError:
        raise RuntimeError("GITHUB_TOKEN için bu deftere erişim izni verilmemiş. 🔑 bölümünde "
                           "'Notebook access' anahtarını aç.") from None
    # Belirteç adrese yazılmaz; yalnız bu komut için başlık olarak verilir, .git/config'e girmez.
    baslik = "Authorization: Basic " + base64.b64encode(f"x-access-token:{belirtec}".encode()).decode()
    sonuc = subprocess.run(
        ["git", "-c", f"http.extraHeader={baslik}", "clone", "--depth", "1", "--branch", DAL, REPO_URL, DEPO_KLASORU],
        capture_output=True, text=True,
    )
    hata = sonuc.stderr.replace(belirtec, "***")
    del belirtec, baslik
    if sonuc.returncode != 0:
        raise RuntimeError("Klonlama başarısız. Belirtecin süresi dolmuş, depo seçilmemiş ya da 'Contents: "
                           "Read-only' izni verilmemiş olabilir.\n" + hata)
    print("Depo klonlandı:", DEPO_KLASORU)
else:
    print("Depo zaten var:", DEPO_KLASORU)

os.chdir(DEPO_KLASORU)

In [ ]:
# 2b. Paketi kur
kurulum = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], capture_output=True, text=True)
if kurulum.returncode != 0:
    print(kurulum.stdout[-3000:], kurulum.stderr[-3000:])
    raise RuntimeError("pip install -e . başarısız")

# Çalışan çekirdek, pip'in yeni eklediği yolu görmez ("No module named 'malzeme_kabul'" hatasının nedeni).
# src/ klasörünü yola elle ekle; böylece çekirdeği yeniden başlatmak gerekmez.
import importlib
SRC = os.path.join(DEPO_KLASORU, "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)
importlib.invalidate_caches()
print("Kurulum tamam")

In [ ]:
# 2c. Kurulumu doğrula
os.chdir(DEPO_KLASORU)
import malzeme_kabul
from malzeme_kabul.yerel_model import YerelModelIstemcisi, cikarim_semasi

assert os.path.realpath(malzeme_kabul.__file__).startswith(DEPO_KLASORU), malzeme_kabul.__file__
# Betikler ayrı süreçte çalışır; orada da paket bulunmalı.
subprocess.run([sys.executable, "-c", "import malzeme_kabul.yerel_model"], check=True)
assert os.path.isfile("data/raw/MIL-PRF-23377K.pdf"), "data/raw altında standart PDF'i yok"
print("Paket:", os.path.dirname(malzeme_kabul.__file__))
print("Sürüm:", subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())
print("Şema alanları:", list(cikarim_semasi()["$defs"]["TestSarti"]["properties"])[:5], "...")
print("✓ Kurulum doğrulandı")

## 3. Ollama'yı kur ve başlat

Sunucu yalnız `127.0.0.1:11434` adresini dinler. Model dosyaları `/content/ollama_modelleri` altına iner; oturum
kapanınca silinir. Her yeni oturumda modeli yeniden indirmek gerekir (7B için 1–2 dakika).

In [ ]:
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import json, subprocess, time, urllib.request

os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"
os.environ["OLLAMA_MODELS"] = "/content/ollama_modelleri"
os.environ["OLLAMA_KEEP_ALIVE"] = "30m"
ollama_sunucu = subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)

for _ in range(60):
    try:
        with urllib.request.urlopen("http://127.0.0.1:11434/api/version", timeout=2) as y:
            print("Ollama hazır:", json.load(y)["version"])
            break
    except OSError:
        time.sleep(1)
else:
    raise RuntimeError("Ollama başlamadı, /content/ollama.log dosyasına bak")

In [ ]:
!ollama pull {MODEL}
!ollama list

## 4. Tek sayfayla dene

Önce bir sayfayı okutup modelin şemaya uygun JSON döndüğünü gör. Aynı istek bütün belgede sayfa sayfa yapılır.

In [ ]:
from malzeme_kabul.kural_cikarici import metni_normallestir
from malzeme_kabul.pdf_okuyucu import pdf_oku
from malzeme_kabul.yerel_model import YerelModelIstemcisi, cikarim_semasi, istem_olustur, yaniti_coz

belge = pdf_oku("data/raw/MIL-PRF-23377K.pdf")
SAYFA = 6   # deneme sayfası; istediğin sayfayı yaz
istemci = YerelModelIstemcisi(model=MODEL, baglam=BAGLAM,
                              dusunme=False if ADAYLAR_7_8B.get(MODEL, {}).get("dusunme_kapali") else None)
mesajlar = istem_olustur(metni_normallestir(belge.sayfalar[SAYFA - 1].metin), SAYFA, "MIL-PRF-23377", "K")

t0 = time.perf_counter()
ham = istemci.sohbet(mesajlar, cikarim_semasi())
print(f"{time.perf_counter() - t0:.1f} s")
testler, hatalar = yaniti_coz(ham)
for t in testler:
    print(t.kaynak.madde, "|", t.test_adi, "|", t.sinir.alt, t.sinir.ust, t.sinir.birim, "|", sorted(t.test_turleri))
print("Hatalar:", hatalar)

## 5. Bütün belgeyi çıkar ve süreyi ölç

`scripts/dil_modeli_cikar.py` her sayfayı modele okutur ve iki dosya yazar:
`data/cikti/dil_modeli/<BELGE>_dil_modeli_<model>.json` (kabul dokümanı, yalnız testler) ve
`..._rapor.json` (sayfa başına süre, hatalar, tekrar eden testler). İlk çağrıda model belleğe yüklendiği için
süreyi ölçmeden önce 4. bölümün çalıştırılmış olması iyi olur.

In [ ]:
def calistir(model, ayar, belgeler=BELGELER):
    ek = ["--dusunme-kapali"] if ayar.get("dusunme_kapali") else []
    for pdf in belgeler:
        subprocess.run(["python", "scripts/dil_modeli_cikar.py", pdf, "--model", model,
                        "--baglam", str(BAGLAM), *ek], check=True)

calistir(MODEL, ADAYLAR_7_8B.get(MODEL, {}))

In [ ]:
import glob
import pandas as pd

def ozet_tablosu():
    satirlar = []
    for yol in sorted(glob.glob("data/cikti/dil_modeli/*_rapor.json")):
        r = json.load(open(yol, encoding="utf-8"))
        sayfa = len(r["sayfalar"])
        satirlar.append({
            "belge": r["belge"], "model": r["model"], "test": r["test_sayisi"],
            "sayfa": sayfa, "hatalı sayfa": sum(1 for s in r["sayfalar"] if s["hatalar"]),
            "tekrar": len(r["tekrar_edenler"]), "belge süresi (s)": r["toplam_sure_s"],
            "sayfa başına (s)": round(r["toplam_sure_s"] / max(sayfa, 1), 1),
        })
    return pd.DataFrame(satirlar)

ozet_tablosu()

In [ ]:
# Hangi sayfada ne hata çıktı? (modelin nerede zorlandığını görmek için)
for yol in sorted(glob.glob("data/cikti/dil_modeli/*_rapor.json")):
    r = json.load(open(yol, encoding="utf-8"))
    for s in r["sayfalar"]:
        for h in s["hatalar"]:
            print(r["model"], "| sayfa", s["sayfa"], "|", h)

### Ön değerlendirme (taslak cevap anahtarıyla)

`data/gold_taslak/MIL-PRF-23377K.json` henüz onaylanmış bir cevap anahtarı değildir; proje sahibi kontrol edip
`data/gold/` altına taşıyana kadar sonuçlar yalnız ön bilgidir. **İstemi ya da çıkarıcıyı bu sonuçlara bakarak
ayarlama**; yoksa sistem cevap anahtarına göre ayarlanmış olur. Raf ömrü, koşullar ve numune hazırlama bu adımda
çıkarılmadığı için o alanların puanı düşük çıkar; testlerle ilgili satırlara bak.

In [ ]:
for yol in sorted(glob.glob("data/cikti/dil_modeli/MIL-PRF-23377K_dil_modeli_*.json")):
    if yol.endswith("_rapor.json"):
        continue
    print("==", yol)
    !python scripts/degerlendir.py {yol} data/gold_taslak/MIL-PRF-23377K.json

## 6. Adayları sırayla dene

Her aday indirilir, bütün belgeler çıkarılır, sonra disk dolmasın diye model silinir. 7–8B adayların her biri
T4'te belge başına birkaç dakika sürer. Dizüstü adaylarının (1–3B) doğruluğu da burada ölçülebilir; dizüstündeki
süre ayrıca ölçülmelidir.

In [ ]:
ADAYLAR = {**ADAYLAR_7_8B}            # dizüstü adaylarını da denemek için: {**ADAYLAR_7_8B, **ADAYLAR_1_3B}
for model, ayar in ADAYLAR.items():
    print("###", model)
    subprocess.run(["ollama", "pull", model], check=True)
    # ısınma: modeli belleğe yükle, süreye katılmasın
    YerelModelIstemcisi(model=model, baglam=BAGLAM, en_cok_token=16,
                        dusunme=False if ayar.get("dusunme_kapali") else None,
                        ).sohbet([{"role": "user", "content": "ok"}], {"type": "object"})
    calistir(model, ayar)
    subprocess.run(["ollama", "rm", model], check=True)

ozet_tablosu()

## 7. (İsteğe bağlı) llama.cpp ile aynı iş

Ollama yerine llama.cpp'nin `llama-server` sunucusu da kullanılabilir. Şema `response_format` içinde
`json_schema` olarak gider ve sunucu onu dilbilgisine (grammar) çevirerek üretimi zorlar. Derleme T4'te 5–10
dakika sürer. Ollama sunucusu çalışıyorsa bellek için önce onu durdur.

In [ ]:
if "ollama_sunucu" in globals():
    ollama_sunucu.terminate()
!git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
!cmake -S /content/llama.cpp -B /content/llama.cpp/build -DGGML_CUDA=ON -DLLAMA_CURL=OFF > /dev/null
!cmake --build /content/llama.cpp/build --config Release -j 4 --target llama-server 2>&1 | tail -2

In [ ]:
from huggingface_hub import hf_hub_download

# Aday GGUF dosyaları (Q4_K_M). Ollama adaylarıyla aynı modeller.
GGUF_ADAYLARI = {
    "qwen2.5-7b": ("bartowski/Qwen2.5-7B-Instruct-GGUF", "Qwen2.5-7B-Instruct-Q4_K_M.gguf"),
    "llama3.1-8b": ("bartowski/Meta-Llama-3.1-8B-Instruct-GGUF", "Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf"),
    "qwen3-8b": ("Qwen/Qwen3-8B-GGUF", "Qwen3-8B-Q4_K_M.gguf"),
}
GGUF = "qwen2.5-7b"
gguf_yolu = hf_hub_download(*GGUF_ADAYLARI[GGUF], local_dir="/content/gguf")

llama_sunucu = subprocess.Popen(
    ["/content/llama.cpp/build/bin/llama-server", "-m", gguf_yolu, "-ngl", "99", "-c", str(BAGLAM),
     "--host", "127.0.0.1", "--port", "8080"],
    stdout=open("/content/llama_server.log", "w"), stderr=subprocess.STDOUT,
)
for _ in range(120):
    try:
        with urllib.request.urlopen("http://127.0.0.1:8080/health", timeout=2) as y:
            if json.load(y).get("status") == "ok":
                print("llama-server hazır")
                break
    except OSError:
        pass
    time.sleep(1)
else:
    raise RuntimeError("llama-server başlamadı, /content/llama_server.log dosyasına bak")

In [ ]:
for pdf in BELGELER:
    !python scripts/dil_modeli_cikar.py {pdf} --model {GGUF}-llamacpp --arka-uc llamacpp --baglam {BAGLAM}
ozet_tablosu()

## 8. Çıktıları indir

Colab oturumu kapanınca dosyalar silinir. Çıktıları zip olarak indir. Depoya eklemek istersen ayrı bir PR ile
`data/cikti/dil_modeli/` altına koy; `data/gold/` klasörüne hiçbir şey yazılmaz.

In [ ]:
!cd /content/malzeme-kabul-ai && zip -qr /content/dil_modeli_ciktilari.zip data/cikti/dil_modeli
from google.colab import files
files.download("/content/dil_modeli_ciktilari.zip")